<a href="https://colab.research.google.com/github/HMXHY/ERFD/blob/main/inference_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ERFD Fake News Detection - Quick Inference Demo
This notebook demonstrates how to automatically download the pre-trained ERFD model and evaluate it on the test set without any manual setup.
We extract the core `Classifier` architecture here for transparency and to avoid Colab runtime argument conflicts.

In [14]:
# 1. Clean environment, clone repo and install dependencies
%cd /content
!rm -rf ERFD
!git clone https://github.com/HMXHY/ERFD.git
%cd ERFD
!pip install -r require4colab.txt

import sys
import os
sys.path.append(os.getcwd())

/content
Cloning into 'ERFD'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (45/45), done.
Receiving objects: 100% (50/50), 22.08 KiB | 7.36 MiB/s, done.
Resolving deltas: 100% (19/19), done.
remote: Total 50 (delta 19), reused 5 (delta 0), pack-reused 0 (from 0)
/content/ERFD


In [15]:
# 2. Download Data and Checkpoints
!gdown --id 1GF3yC8hKWIDwrxJvZYgTHkzsv8BBhgOH
!unzip -o ERFD_demo_files.zip
print("Data and checkpoints successfully loaded!")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1GF3yC8hKWIDwrxJvZYgTHkzsv8BBhgOH
From (redirected): https://drive.google.com/uc?id=1GF3yC8hKWIDwrxJvZYgTHkzsv8BBhgOH&confirm=t&uuid=1f331945-2b69-41cf-b256-d0886d05a1db
To: /content/ERFD/ERFD_demo_files.zip
100% 391M/391M [00:06<00:00, 55.9MB/s]
Archive:  ERFD_demo_files.zip
   creating: data_ids/
   creating: data_ids/adversarial_test/
  inflating: data_ids/adversarial_test/covid_test_adv_A.pkl  
  inflating: data_ids/adversarial_test/covid_test_adv_B.pkl  
  inflating: data_ids/adversarial_test/covid_test_adv_C.pkl  
  inflating: data_ids/adversarial_test/covid_test_adv_D.pkl  
  inflating: data_ids/adversarial_test/gossipcop_test_adv_A.pkl  
  inflating: data_ids/adversarial_test/gossipcop_test_adv_B.pk

In [16]:
# 3. Extract Core Model Architecture for Demo (Bypassing training scripts)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Import submodules safely
from models.bert import RobertaClassifier
from models.fourierattention import FourierAttention
from utils.load_graphdata import load_origindata_test
from result_output.log_result import output_metrics_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Classifier(nn.Module):
    def __init__(self, hidden_dim, freq_dim, attn_heads, dropnum):
        super().__init__()
        self.bert = RobertaClassifier()
        self.fourier_attn = FourierAttention(hidden_dim=hidden_dim, attn_heads=attn_heads)
        self.dropout = nn.Dropout(p=dropnum)
        self.out_layer = nn.Sequential(
            nn.Linear(hidden_dim+freq_dim, 256),
            nn.ReLU(),
            nn.Linear(256,2)
        )
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim*2, hidden_dim),
            nn.Sigmoid()
        )
        self.aigc_head = nn.Sequential(
            nn.Dropout(dropnum),
            nn.Linear(hidden_dim+freq_dim, 256),
            nn.ReLU(),
            nn.Linear(256,2)
        )
        self.freq_lowdim = nn.Linear(hidden_dim, freq_dim)
        self.hidden_dim = hidden_dim

    def forward(self, input_ids, attention_masks):
        seq_feat = self.bert(input_ids = input_ids, attention_mask = attention_masks)
        freq_feat = self.fourier_attn(seq_feat.last_hidden_state)
        gate = self.gate(torch.cat([freq_feat, seq_feat[1]], dim=-1))
        weight_freq_feat = self.freq_lowdim(gate * freq_feat)
        gated_output = torch.cat([weight_freq_feat, (1-gate) * seq_feat[1]], dim=-1)
        logit = self.out_layer(self.dropout(gated_output))
        aigc_logits = self.aigc_head(gated_output)
        return logit, aigc_logits

class Testset(Dataset):
    def __init__(self, input_ids, masks, label, max_len):
        self.input_ids = input_ids
        self.masks = masks
        self.label = label
        self.max_len = max_len
    def __getitem__(self, item):
        return {
            'input_ids': self.input_ids[item],
            'attention_mask': self.masks[item],
            'label': torch.tensor(self.label[item], dtype=torch.long),
            'idx': item
        }
    def __len__(self):
        return self.input_ids.size(0)

def create_eval_loader(input_ids, masks, label, max_len, batch_size):
    ds = Testset(input_ids, masks, np.array(label), max_len)
    return DataLoader(ds, batch_size=batch_size, num_workers=0)

def test_ouput(test_loader, model):
    y_pred, y_test = [], []
    for Batch_data in tqdm(test_loader):
        with torch.no_grad():
            input_ids = Batch_data['input_ids'].to(device)
            attention_mask = Batch_data['attention_mask'].to(device)
            targets = Batch_data['label'].to(device)
            val_out, _ = model(input_ids=input_ids, attention_masks=attention_mask)
            _, val_pred = val_out.max(dim=1)
            y_pred.append(val_pred)
            y_test.append(targets)
    return y_test, y_pred

In [17]:
# 4. Mock Arguments for Jupyter Environment
class Args:
    dataset_name = 'politifact'
    hidden_dim = 768
    freq_dim = 4
    attn_heads = 1
    dropout_num = 0.4
    max_len = 512
    batch_size = 4
args = Args()

# 5. Load All Test Data (Origin + Reframings A, B, C, D)
print("Loading All Test Data (Origin, A, B, C, D)...")
test_input_ids, test_masks, test_label = load_origindata_test(args.dataset_name)

test_loader_O = create_eval_loader(test_input_ids['O'], test_masks['O'], test_label, args.max_len, args.batch_size)
test_loader_A = create_eval_loader(test_input_ids['A'], test_masks['A'], test_label, args.max_len, args.batch_size)
test_loader_B = create_eval_loader(test_input_ids['B'], test_masks['B'], test_label, args.max_len, args.batch_size)
test_loader_C = create_eval_loader(test_input_ids['C'], test_masks['C'], test_label, args.max_len, args.batch_size)
test_loader_D = create_eval_loader(test_input_ids['D'], test_masks['D'], test_label, args.max_len, args.batch_size)

# 6. Load Pre-trained Model (with strict=False to ignore unused variables)
print("\nLoading Pre-trained Model Parameters...")
model = Classifier(args.hidden_dim, args.freq_dim, args.attn_heads, args.dropout_num).to(device)
model.load_state_dict(torch.load('checkpoints/ERFD/politifact_iter0.m', map_location=device), strict=False)
model.eval()

# 7. Run Evaluation on All Sets
print("\nRunning Inference on All Test Sets...")
y_test, y_pred_O = test_ouput(test_loader_O, model)
_, y_pred_A = test_ouput(test_loader_A, model)
_, y_pred_B = test_ouput(test_loader_B, model)
_, y_pred_C = test_ouput(test_loader_C, model)
_, y_pred_D = test_ouput(test_loader_D, model)

combined_pred = y_pred_A + y_pred_B + y_pred_C + y_pred_D
combined_true = y_test + y_test + y_test + y_test

# 8. Print Results Clearly
print("\n" + "="*50)
print("FINAL EVALUATION RESULTS")
print("="*50)

res_O = output_metrics_metrics(y_test, y_pred_O, 'Origin')
res_A = output_metrics_metrics(y_test, y_pred_A, 'Test A')
res_B = output_metrics_metrics(y_test, y_pred_B, 'Test B')
res_C = output_metrics_metrics(y_test, y_pred_C, 'Test C')
res_D = output_metrics_metrics(y_test, y_pred_D, 'Test D')
res_Comb = output_metrics_metrics(combined_true, combined_pred, 'Combined')

def print_metrics(name, metrics):
    print(f"{name:<10} | Acc: {metrics[0]:.2f}% | Prec: {metrics[1]:.2f}% | Rec: {metrics[2]:.2f}% | F1: {metrics[3]:.2f}%")

print_metrics('Origin', res_O)
print_metrics('Test A', res_A)
print_metrics('Test B', res_B)
print_metrics('Test C', res_C)
print_metrics('Test D', res_D)
print_metrics('Combined', res_Comb)
print("="*50)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading All Test Data (Origin, A, B, C, D)...
politifact test set load

Loading Pre-trained Model Parameters...

Running Inference on All Test Sets...


100%|██████████| 23/23 [02:25<00:00,  6.31s/it]


FINAL EVALUATION RESULTS
Origin     | Acc: 94.44% | Prec: 94.64% | Rec: 94.44% | F1: 94.44%
Test A     | Acc: 80.00% | Prec: 80.06% | Rec: 80.00% | F1: 79.99%
Test B     | Acc: 78.89% | Prec: 78.90% | Rec: 78.89% | F1: 78.89%
Test C     | Acc: 85.56% | Prec: 85.71% | Rec: 85.56% | F1: 85.54%
Test D     | Acc: 84.44% | Prec: 84.72% | Rec: 84.44% | F1: 84.41%
Combined   | Acc: 82.22% | Prec: 82.24% | Rec: 82.22% | F1: 82.22%
